In [1]:
import pandas as pd
import spacy
from collections import Counter
import plotly.express as px
from tqdm import tqdm
import os
import numpy as np

# Load
DATASET_PATH = "/ltstorage/home/4xin/uhh-ias-ml/data/processed/final_combined_jokes.csv"
df = pd.read_csv(DATASET_PATH, dtype=str)
SAVE_PATH = "./ouputs"
os.makedirs(SAVE_PATH, exist_ok=True)

# 1. Count all tag types

For each joke, we generate its tags by:
1. Check whether the joke contains any NOUN or PROPN. If so, use the top 3 (or fewer) of them as the tags.
2. If no NOUN or PROPN is found, check for VERBS or ADJS. If any are present, use the top 3 (or fewer) of them as the tags.
3. If none of the above types are found, assign the tag "misc".

This is the tag-generation logic for the "topic" column in "final_combined_jokes.csv"

In [ ]:
def map_pos(pos):
    if pos in ["NOUN"]:
        return "NOUN"
    elif pos in ["PROPN"]:
        return "PROPN"
    elif pos in ["VERB"]:
        return "VERB"
    elif pos in ["ADJ"]:
        return "ADJ"
    else:
        return "MISC"

def count_tag_types(df):
    # Load Spacy English model
    nlp = spacy.load("en_core_web_sm")
    tag_types = {
        "NOUN": 0,
        "PROPN": 0,
        "VERB": 0,
        "ADJ": 0,
        "MISC": 0
    }
    for idx, t in tqdm(df["topic"].items()):
        t = str(t).strip()
        if t == "misc":
            tag_types["MISC"] += 1
            continue
        tags = [tag.strip().lower() for tag in t.split(", ") if tag.strip()]
        for tag in tags:
            pos = nlp(tag)[0].pos_
            tag_type = map_pos(pos)
            tag_types[tag_type] += 1
    print(tag_types)
    # {'NOUN': 1300645, 'PROPN': 424317, 'VERB': 175586, 'ADJ': 57853, 'MISC': 28505, "NULL": 0}
    tag_type_df = pd.DataFrame(list(tag_types.items()), columns=["Type", "Count"])
    tag_type_df.to_csv(f"{SAVE_PATH}/tag_type_count.csv", index=False)
    return tag_types

def plot_tag_type_count():
    STAT_PATH = "ouputs/tag_type_count.csv"
    df = pd.read_csv(STAT_PATH)

    # Convert count to int
    df["Count"] = df["Count"].astype(int)

    # Compute percentage
    total = df["Count"].sum()
    df["Percentage"] = df["Count"] / total * 100

    # Text displayed on top of bar (count + percent)
    df["Label"] = df.apply(
        lambda row: f"{row['Count']:,} ({row['Percentage']:.1f}%)",
        axis=1
    )

    fig = px.bar(
        df,
        x="Type",
        y="Count",
        text="Label",             # show both count + %
        title="Distribution of Tag Types (POS Categories)",
        color="Type",
        color_discrete_sequence=px.colors.qualitative.Safe,
        hover_data={
            "Count": True,
            "Percentage": ":.2f",  # format percentage
            "Type": True,
            "Label": False
        }
    )

    fig.update_traces(
        textposition="outside",
        hovertemplate=
            "Type: %{x}<br>" +
            "Count: %{y:,}<br>" +
            "Percentage: %{customdata[0]:.2f}%<extra></extra>"
    )

    fig.update_layout(
        title_x=0.5,
        xaxis_title="Tag Type (POS)",
        yaxis_title="Count",
        bargap=0.25,
        yaxis=dict(type="linear", autorange=True)
    )

    fig.write_html(f"{SAVE_PATH}/tag_type_count.html")
    fig.write_image(f"{SAVE_PATH}/tag_type_count.png")
    fig.show()

In [13]:
# count_tag_types(df)
plot_tag_type_count()

# 2. Plot the distribution of joke lengths

In [4]:
def joke_length(df):
    # Compute joke lengths (characters)
    df_len = pd.DataFrame({
    "joke_length_chars": df["joke"].apply(len)
    })
    df_len.to_csv(f"{SAVE_PATH}/joke_length_chars.csv", index=False)
    return df_len

def plot_joke_length_distribution():
    STAT_PATH = "/ltstorage/home/4xin/uhh-ias-ml/data/ouputs/joke_length_chars.csv"
    df_len = pd.read_csv(STAT_PATH)
    df_len["joke_length_chars"] = df_len["joke_length_chars"].astype(int)

    lengths = df_len["joke_length_chars"]

    # 1. Build bins with width = 10
    min_len = lengths.min()
    max_len = lengths.max()
    start = (min_len // 10) * 10
    end = ((max_len // 10) + 1) * 10
    bin_edges = np.arange(start, end + 1, 10)

    # 2. Histogram
    counts, edges = np.histogram(lengths, bins=bin_edges)
    bin_left = edges[:-1]
    bin_right = edges[1:] - 1      # inclusive right side
    bin_center = (bin_left + bin_right) / 2.0

    # 3. Cumulative percentage
    cumulative = np.cumsum(counts)
    cumulative_percent = cumulative / cumulative[-1] * 100

    # 4. Build DataFrame
    df_hist = pd.DataFrame({
        "bin_left": bin_left,
        "bin_right": bin_right,
        "bin_center": bin_center,
        "count": counts,
        "cumulative_percent": cumulative_percent
    })
    df_hist["bin_label"] = df_hist.apply(
        lambda row: f"{int(row['bin_left'])}–{int(row['bin_right'])}", axis=1
    )

    # 5. Bar chart (x is numeric center; tick labels are the ranges)
    fig = px.bar(
        df_hist,
        x="bin_center",
        y="count",
        color="cumulative_percent",
        color_continuous_scale="Magma",  # or Purpor / Plasma if you prefer
        custom_data=["bin_left", "bin_right", "cumulative_percent"],
        labels={
            "bin_center": "Joke Length (characters, grouped by 10)",
            "count": "Count",
            "cumulative_percent": "Cumulative (%)"
        },
        title="Distribution of Joke Lengths (Characters) (Bin width = 10)"
    )

    # Use pretty range labels on the x-axis
    step = 3
    idx = np.arange(0, len(df_hist), step)

    fig.update_xaxes(
        tickmode="array",
        tickvals=df_hist.loc[idx, "bin_center"],
        ticktext=df_hist.loc[idx, "bin_label"],
    )

    # Hover displays range + count + cumulative
    fig.update_traces(
        hovertemplate=(
            "Length = %{customdata[0]:.0f} – %{customdata[1]:.0f}<br>"
            "Count = %{y}<br>"
            "Cumulative = %{customdata[2]:.2f}%<extra></extra>"
        )
    )

    # 6. Find the first bin where cumulative >= 90%
    mask_90 = df_hist["cumulative_percent"] >= 90
    idx_90 = df_hist.index[mask_90][0]  # first index satisfying condition

    bin_left_90 = df_hist.loc[idx_90, "bin_left"]
    bin_right_90 = df_hist.loc[idx_90, "bin_right"]
    cum_90 = df_hist.loc[idx_90, "cumulative_percent"]
    x_90 = df_hist.loc[idx_90, "bin_center"]

    # Add vertical line at that x
    fig.add_vline(
        x=x_90,
        line_width=1,
        # line_dash="dash",
        line_color="red"
    )

    # Add annotation near the top
    fig.add_annotation(
        x=x_90,
        y=df_hist["count"].max(),
        text=f"90% cumulative<br>Length ≈ {bin_left_90}–{bin_right_90}",
        showarrow=True,
        arrowhead=2,
        arrowsize=1,
        arrowcolor="red",
        font=dict(color="red", size=12),
        yshift=20
    )

    fig.update_layout(
        title_x=0.5,
        bargap=0
    )

    fig.write_html(f"{SAVE_PATH}/joke_length_chars.html")
    fig.write_image(f"{SAVE_PATH}/joke_length_chars.png")
    fig.show()


In [14]:
# joke_length(df)
plot_joke_length_distribution()

# 3. Plot the distribution of number of tags per joke

In [17]:
def count_tags_per_joke(df):
    tag_count_per_joke = {
        "1": 0,
        "2": 0,
        "3": 0,
    }
    for t in df["topic"]:
        t = str(t).strip()
        if t == "misc":
            tag_count_per_joke["1"] += 1
            continue
        tags = [tag.strip().lower() for tag in t.split(", ") if tag.strip()]
        l = len(tags)
        tag_count_per_joke[str(l)] += 1
    print(tag_count_per_joke)
    # {'1': 42498, '2': 108457, '3': 575832}
    df_tags = pd.DataFrame(list(tag_count_per_joke.items()), columns=["tag count", "number of jokes"])
    df_tags.to_csv(f"{SAVE_PATH}/tag_count_distribution.csv", index=False)
    return df_tags

def plot_tag_count_distribution():
    STAT_PATH = "/ltstorage/home/4xin/uhh-ias-ml/data/ouputs/tag_count_distribution.csv"
    df_tags = pd.read_csv(STAT_PATH)

    # Ensure correct dtypes
    df_tags["Tag Count"] = df_tags["Tag Count"].astype(str)
    df_tags["Number of Jokes"] = df_tags["Number of Jokes"].astype(int)

    # Percentage for each tag-count bucket
    total = df_tags["Number of Jokes"].sum()
    df_tags["Percent"] = df_tags["Number of Jokes"] / total * 100

    # Label on top of each bar: "count (xx.x%)"
    df_tags["LabelText"] = df_tags.apply(
        lambda row: f"{row['Number of Jokes']:,} ({row['Percent']:.1f}%)",
        axis=1
    )

    fig = px.bar(
        df_tags,
        x="Tag Count",
        y="Number of Jokes",
        text="LabelText",
        title="Number of Tags per Joke",
        color_discrete_sequence=["#636EFA"],
        custom_data=["Percent"]   # pass percentage to hovertemplate
    )

    fig.update_traces(
        textposition="outside",
        textfont=dict(size=12),
        hovertemplate=(
            "Tag Count: %{x}<br>"
            "Number of Jokes: %{y:,}<br>"
            "Percentage: %{customdata[0]:.2f}%<extra></extra>"
        )
    )

    fig.update_layout(
        showlegend=False,
        title_x=0.5,
        xaxis_title="Number of Tags per Joke",
        yaxis_title="Number of Jokes",
        bargap=0.3,
        margin=dict(b=80)
    )

    fig.write_html(f"{SAVE_PATH}/tag_count_distribution.html")
    fig.write_image(f"{SAVE_PATH}/tag_count_distribution.png")
    fig.show()


In [18]:
# count_tags_per_joke(df)
plot_tag_count_distribution()

# 4. Plot top n tags

In [21]:
def count_tag_frequencies(df):
    # Count the frequency of each topic tag across all jokes.
    all_tags = []

    for t in tqdm(df["topic"]):
        if not isinstance(t, str):
            continue
        t = t.strip()
        tags = [tag.strip().lower() for tag in t.split(",") if tag.strip()]
        all_tags.extend(tags)

    tag_counter = Counter(all_tags)
    tag_freq_df = pd.DataFrame(tag_counter.items(), columns=["tag", "count"]).sort_values(
        "count", ascending=False
    ).reset_index(drop=True)

    tag_freq_df.to_csv(f"{SAVE_PATH}/tag_frequencies_full.csv", index=False)

    return tag_freq_df

def topn_wo_other(n=100):
    """
    Keep top-n rows; sum the rest as a single 'other' row.
    Ensures 'other' is last (if it exists).
    """
    STAT_PATH = "/ltstorage/home/4xin/uhh-ias-ml/data/ouputs/tag_frequencies_full.csv"
    freq_df = pd.read_csv(STAT_PATH)

    if len(freq_df) <= n:
        return freq_df.copy()

    top = freq_df.iloc[:n].copy()
    output = top
    output.to_csv(f"{SAVE_PATH}/tag_frequencies_top{n}.csv", index=False)
    return output

def plot_tag_frequencies_log(n=100):
    STAT_PATH = f"/ltstorage/home/4xin/uhh-ias-ml/data/ouputs/tag_frequencies_top{n}.csv"
    df_plot = pd.read_csv(STAT_PATH)

    title = f"Top {n} Tags"

    fig = px.bar(
        df_plot,
        x="tag",
        y="count",
        text="count",
        title=title,
        color_discrete_sequence=["#636EFA"]
    )

    fig.update_traces(
        textposition="outside",
        opacity=1.0
    )

    # ---- Y-axis log scale ----
    fig.update_yaxes(
        type="log",
        tickmode="array",
        tickvals=[0, 5000, 10000, 15000, 20000, 25000, 30000],
        ticktext=["0", "5k", "10k", "15k", "20k", "25k", "30k"],
        title="Frequency (log scale)"
    )

    fig.update_layout(
        xaxis_title="Tag",
        bargap=0,
        title_x=0.5
    )

    fig.write_html(f"{SAVE_PATH}/tag_frequencies_top{n}.html")
    fig.write_image(f"{SAVE_PATH}/tag_frequencies_top{n}.png")
    fig.show()




## 4.1. n = 30

In [22]:
# topn_wo_other(n=30)
plot_tag_frequencies_log(n=30)

## 4.2. n=100

In [23]:
# topn_wo_other(n=100)
plot_tag_frequencies_log(n=100)

## 4.3. n=500

In [24]:
# topn_wo_other(n=500)
plot_tag_frequencies_log(n=500)

## 4.4. n=1000

In [25]:
# topn_wo_other(n=1000)
plot_tag_frequencies_log(n=1000)

# 5. Plot the distribution of tag frequencies

In [36]:
def plot_tag_frequency_distribution_log():
    STAT_PATH = "/ltstorage/home/4xin/uhh-ias-ml/data/ouputs/tag_frequencies_full.csv"
    df = pd.read_csv(STAT_PATH)

    freqs = df["count"].astype(int)
    freqs = freqs[freqs > 0]

    # Define log bins (powers of 2)
    bins = 2 ** np.arange(0, 20)  # 1, 2, 4, 8, ..., 32768

    counts, edges = np.histogram(freqs, bins=bins)

    # Filter out empty bins
    valid = counts > 0
    counts = counts[valid]
    left = edges[:-1][valid]
    right = edges[1:][valid] - 1

    # Create labels like "4–7", "8–15", etc.
    labels = [f"{l}–{r}" for l, r in zip(left, right)]

    total = counts.sum()
    percent = counts / total * 100

    df_hist = pd.DataFrame({
        "freq_range": labels,
        "count": counts,
        "percent": percent
    })

    # Label on top of bars: "123 (4.5%)"
    df_hist["LabelText"] = df_hist.apply(
        lambda row: f"{row['count']:,} ({row['percent']:.1f}%)", axis=1
    )

    fig = px.bar(
        df_hist,
        x="freq_range",
        y="count",
        text="LabelText",
        title="Distribution of Tag Frequencies (Log-Binned)",
        color_discrete_sequence=["#636EFA"],
        custom_data=["percent"]  # for hover
    )

    fig.update_traces(
        textposition="outside",
        textfont=dict(size=11),
        hovertemplate=(
            "Frequency Range: %{x}<br>"
            "Number of Tags: %{y:,}<br>"
            "Percentage: %{customdata[0]:.2f}%"
            "<extra></extra>"
        )
    )

    fig.update_layout(
        title_x=0.5,
        xaxis_title="Tag Frequency Range (log scale)",
        yaxis_title="Number of Tags",
        bargap=0.15,
        margin=dict(b=120)  # avoid x-label overlap
    )

    fig.write_html(f"{SAVE_PATH}/tag_frequency_distribution_log.html")
    fig.write_image(f"{SAVE_PATH}/tag_frequency_distribution_log.png")
    fig.show()

In [37]:
plot_tag_frequency_distribution_log()

In [6]:
import pandas as pd
import spacy
from collections import Counter
import plotly.express as px
from tqdm import tqdm
import os
import numpy as np

# Load
DATASET_PATH = "/ltstorage/home/4xin/uhh-ias-ml/data/processed/final_clean_jokes_with_all_nouns.csv"
df = pd.read_csv(DATASET_PATH, dtype=str)
SAVE_PATH = "./ouputs"
os.makedirs(SAVE_PATH, exist_ok=True)

In [10]:
def count_unique_tags_per_joke(df, topic_col="topic_all_nouns"):
    """
    Count how many UNIQUE tags each joke has, based on `topic_col`,
    and aggregate a distribution: Tag Count -> Number of Jokes.
    Also saves the distribution as CSV.
    """
    tag_count_per_joke = {}   # key: unique tag count (int), value: number of jokes
    unique_counts = []        # keep per-joke counts if you want to inspect later

    for t in df[topic_col]:
        t = str(t).strip()
        if t == "":
            n_unique = 0
        else:
            tags = [tag.strip().lower() for tag in t.split(",") if tag.strip()]
            n_unique = len(set(tags))   # unique tag count

        unique_counts.append(n_unique)
        tag_count_per_joke[n_unique] = tag_count_per_joke.get(n_unique, 0) + 1

    # Aggregate distribution -> DataFrame
    df_tags = pd.DataFrame(
        {
            "Tag Count": list(tag_count_per_joke.keys()),
            "Number of Jokes": list(tag_count_per_joke.values()),
        }
    ).sort_values("Tag Count")

    # Save distribution
    df_tags.to_csv(f"{SAVE_PATH}/tag_count_distribution_all_nouns.csv", index=False)
    return df_tags


def plot_unique_tag_count_distribution():
    STAT_PATH = f"{SAVE_PATH}/tag_count_distribution_all_nouns.csv"
    df_tags = pd.read_csv(STAT_PATH)

    # Ensure correct dtypes
    df_tags["Tag Count"] = df_tags["Tag Count"].astype(int)
    df_tags["Number of Jokes"] = df_tags["Number of Jokes"].astype(int)

    # Percentage for each tag-count bucket
    total = df_tags["Number of Jokes"].sum()
    df_tags["Percent"] = df_tags["Number of Jokes"] / total * 100

    # Label on top of each bar: "count (xx.x%)"
    df_tags["LabelText"] = df_tags.apply(
        lambda row: f"{row['Number of Jokes']:,} ({row['Percent']:.1f}%)",
        axis=1
    )

    fig = px.bar(
        df_tags,
        x="Tag Count",
        y="Number of Jokes",
        text="LabelText",
        title="Number of Unique Tags per Joke (all-nouns topics)",
        color_discrete_sequence=["#636EFA"],
        custom_data=["Percent"]
    )

    fig.update_traces(
        textposition="outside",
        textfont=dict(size=12),
        hovertemplate=(
            "Tag Count: %{x}<br>"
            "Number of Jokes: %{y:,}<br>"
            "Percentage: %{customdata[0]:.2f}%<extra></extra>"
        )
    )

    fig.update_layout(
        showlegend=False,
        title_x=0.5,
        xaxis_title="Number of Unique Tags per Joke",
        yaxis_title="Number of Jokes",
        bargap=0.2,
        margin=dict(b=80)
    )

    fig.write_html(f"{SAVE_PATH}/tag_count_distribution_all_nouns.html")
    fig.write_image(f"{SAVE_PATH}/tag_count_distribution_all_nouns.png")
    fig.show()

In [13]:
count_unique_tags_per_joke(df, topic_col="topic_all_nouns")
plot_unique_tag_count_distribution()

# 7. Visualize the average joke length for different numbers of unique tags

In [11]:
def compute_tag_count_length_stats(df, topic_col="topic_all_nouns", text_col="joke_cleaned"):
    """
    For each joke:
      - compute the number of UNIQUE tags in `topic_col`
      - compute its length in characters from `text_col`

    Then group by the unique tag count and compute:
      - number_of_jokes
      - avg_joke_length
      - std_joke_length

    Tag count = 0 (no tags) is also recorded and kept in the output.
    """
    unique_counts = []
    lengths = []

    for tags, txt in zip(df[topic_col], df[text_col]):
        # Normalize tags
        if isinstance(tags, str):
            tags_str = tags.strip()
        else:
            tags_str = ""

        # Treat empty string or "nan" as no tags
        if tags_str == "" or tags_str.lower() == "nan":
            n_unique = 0
        else:
            tag_list = [t.strip().lower() for t in tags_str.split(",") if t.strip()]
            n_unique = len(set(tag_list))

        # Normalize text and compute length
        txt = "" if not isinstance(txt, str) else txt
        joke_len = len(txt)

        unique_counts.append(n_unique)
        lengths.append(joke_len)

    df = df.copy()
    df["unique_tag_count"] = unique_counts
    df["joke_length_chars"] = lengths

    grouped = (
        df.groupby("unique_tag_count")
          .agg(
              number_of_jokes=("joke_length_chars", "size"),
              avg_joke_length=("joke_length_chars", "mean"),
              std_joke_length=("joke_length_chars", "std")
          )
          .reset_index()
          .rename(columns={"unique_tag_count": "tag_count"})
          .sort_values("tag_count")
    )

    grouped["avg_joke_length"] = grouped["avg_joke_length"].round(2)
    grouped["std_joke_length"] = grouped["std_joke_length"].fillna(0).round(2)

    grouped.to_csv(f"{SAVE_PATH}/tag_count_length_stats_clean.csv", index=False)
    return grouped

In [ ]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go

def plot_tag_count_vs_length():
    STAT_PATH = f"{SAVE_PATH}/tag_count_length_stats_clean.csv"
    df_stats = pd.read_csv(STAT_PATH)

    df_stats["tag_count"] = df_stats["tag_count"].astype(int)
    df_stats["number_of_jokes"] = df_stats["number_of_jokes"].astype(int)
    df_stats["avg_joke_length"] = df_stats["avg_joke_length"].astype(float)
    df_stats["std_joke_length"] = df_stats["std_joke_length"].astype(float)

    # Sort explicitly by tag_count to ensure correct ordering (including 0)
    df_stats = df_stats.sort_values("tag_count")

    x = df_stats["tag_count"]
    cumulative = np.cumsum(df_stats["number_of_jokes"])
    cumulative_percent = cumulative / cumulative[-1] * 100
    print(cumulative, "\n"< cumulative_percent)
    
    mean = df_stats["avg_joke_length"]
    std = df_stats["std_joke_length"]

    upper = mean + std
    lower = mean - std

    fig = make_subplots(
        specs=[[{"secondary_y": True}]],
        shared_xaxes=True
    )

    # --- Left Y: Number of jokes (bar) ---
    fig.add_trace(
        go.Bar(
            x=x,
            y=df_stats["number_of_jokes"],
            name="Number of Jokes",
            marker_color="#636EFA",
            text=[f"{v:,}" for v in df_stats["number_of_jokes"]],
            textposition="outside"
        ),
        secondary_y=False
    )

    # --- Std shaded band on right Y ---
    fig.add_trace(
        go.Scatter(
            x=list(x) + list(x[::-1]),
            y=list(upper) + list(lower[::-1]),
            fill="toself",
            fillcolor="rgba(239,85,59,0.3)",
            line=dict(color="rgba(0,0,0,0)"),
            hoverinfo="skip",
            name="Std Dev"
        ),
        secondary_y=True
    )

    # --- Mean line (with std passed in customdata) ---
    fig.add_trace(
        go.Scatter(
            x=x,
            y=mean,
            name="Average Joke Length",
            mode="lines+markers",
            marker=dict(size=7),
            line=dict(width=2, color="#EF553B"),
            customdata=np.stack([std], axis=-1)  # shape (n, 1)
        ),
        secondary_y=True
    )

    fig.update_layout(
        title="Tag Count vs Number of Jokes and Average Joke Length (with Std Dev)",
        title_x=0.5,
        xaxis_title="Number of Unique Tags per Joke",
        bargap=0.2,
        legend=dict(
            orientation="h",
            yanchor="bottom",
            y=1.02,
            xanchor="center",
            x=0.5
        ),
        margin=dict(b=80)
    )

    fig.update_yaxes(title_text="Number of Jokes", secondary_y=False)
    fig.update_yaxes(title_text="Average Joke Length (chars)", secondary_y=True)

    # Hover for the mean line
    fig.update_traces(
        hovertemplate=(
            "Tag Count: %{x}<br>"
            "Avg Length: %{y:.2f} chars<br>"
            "Std Dev: %{customdata[0]:.2f}<extra></extra>"
        ),
        selector=dict(type="scatter", name="Average Joke Length")
    )

    fig.write_html(f"{SAVE_PATH}/tag_count_vs_length_with_std_clean.html")
    fig.write_image(f"{SAVE_PATH}/tag_count_vs_length_with_std_clean.png")
    fig.show()

In [15]:
compute_tag_count_length_stats(df, topic_col="topic_all_nouns", text_col="joke_cleaned")
plot_tag_count_vs_length()

In [4]:
def joke_length(df):
    # Compute joke lengths (characters)
    df_len = pd.DataFrame({
    "joke_length_chars": df["joke"].apply(len)
    })
    df_len.to_csv(f"{SAVE_PATH}/joke_length_chars.csv", index=False)
    return df_len

def plot_joke_length_distribution():
    STAT_PATH = "/ltstorage/home/4xin/uhh-ias-ml/data/ouputs/joke_length_chars.csv"
    df_len = pd.read_csv(STAT_PATH)
    df_len["joke_length_chars"] = df_len["joke_length_chars"].astype(int)

    lengths = df_len["joke_length_chars"]

    # 1. Build bins with width = 10
    min_len = lengths.min()
    max_len = lengths.max()
    start = (min_len // 10) * 10
    end = ((max_len // 10) + 1) * 10
    bin_edges = np.arange(start, end + 1, 10)

    # 2. Histogram
    counts, edges = np.histogram(lengths, bins=bin_edges)
    bin_left = edges[:-1]
    bin_right = edges[1:] - 1      # inclusive right side
    bin_center = (bin_left + bin_right) / 2.0

    # 3. Cumulative percentage
    cumulative = np.cumsum(counts)
    cumulative_percent = cumulative / cumulative[-1] * 100

    # 4. Build DataFrame
    df_hist = pd.DataFrame({
        "bin_left": bin_left,
        "bin_right": bin_right,
        "bin_center": bin_center,
        "count": counts,
        "cumulative_percent": cumulative_percent
    })
    df_hist["bin_label"] = df_hist.apply(
        lambda row: f"{int(row['bin_left'])}–{int(row['bin_right'])}", axis=1
    )

    # 5. Bar chart (x is numeric center; tick labels are the ranges)
    fig = px.bar(
        df_hist,
        x="bin_center",
        y="count",
        color="cumulative_percent",
        color_continuous_scale="Magma",  # or Purpor / Plasma if you prefer
        custom_data=["bin_left", "bin_right", "cumulative_percent"],
        labels={
            "bin_center": "Joke Length (characters, grouped by 10)",
            "count": "Count",
            "cumulative_percent": "Cumulative (%)"
        },
        title="Distribution of Joke Lengths (Characters) (Bin width = 10)"
    )

    # Use pretty range labels on the x-axis
    step = 3
    idx = np.arange(0, len(df_hist), step)

    fig.update_xaxes(
        tickmode="array",
        tickvals=df_hist.loc[idx, "bin_center"],
        ticktext=df_hist.loc[idx, "bin_label"],
    )

    # Hover displays range + count + cumulative
    fig.update_traces(
        hovertemplate=(
            "Length = %{customdata[0]:.0f} – %{customdata[1]:.0f}<br>"
            "Count = %{y}<br>"
            "Cumulative = %{customdata[2]:.2f}%<extra></extra>"
        )
    )

    # 6. Find the first bin where cumulative >= 90%
    mask_90 = df_hist["cumulative_percent"] >= 90
    idx_90 = df_hist.index[mask_90][0]  # first index satisfying condition

    bin_left_90 = df_hist.loc[idx_90, "bin_left"]
    bin_right_90 = df_hist.loc[idx_90, "bin_right"]
    cum_90 = df_hist.loc[idx_90, "cumulative_percent"]
    x_90 = df_hist.loc[idx_90, "bin_center"]

    # Add vertical line at that x
    fig.add_vline(
        x=x_90,
        line_width=1,
        # line_dash="dash",
        line_color="red"
    )

    # Add annotation near the top
    fig.add_annotation(
        x=x_90,
        y=df_hist["count"].max(),
        text=f"90% cumulative<br>Length ≈ {bin_left_90}–{bin_right_90}",
        showarrow=True,
        arrowhead=2,
        arrowsize=1,
        arrowcolor="red",
        font=dict(color="red", size=12),
        yshift=20
    )

    fig.update_layout(
        title_x=0.5,
        bargap=0
    )

    fig.write_html(f"{SAVE_PATH}/joke_length_chars.html")
    fig.write_image(f"{SAVE_PATH}/joke_length_chars.png")
    fig.show()
